In [ ]:
import networkx as nx
import numpy as np

In [ ]:
def get_trees(A, P, root_cell):
    G = nx.from_numpy_array(A, create_using=nx.DiGraph)
    trees = []
    for i in range(P.shape(1)):
        prob = P[root_cell][i]
        for u in list(G.predecessors(root_cluster)):
            G.remove_edge(u, root_cluster)
        msa = nx.maximum_spanning_arborescence(G)
        trees.append((prob, msa))
    
    return trees

def get_pseudotime(trees, target_cell, P):
    for i in range(P.shape(1)):
        prob = P[target_cell][i]




In [ ]:
def get_path_from_root(tree, root, target):
    msa = tree[1]
    path = [target]
    current = target
    
    while current != root:
        preds = list(msa.predecessors(current))
        
        if not preds:
            raise ValueError("Target not reachable from root")
        
        current = preds[0]
        path.append(current)
    
    path.reverse()
    return path

def get_chain_from_path(path, x_out):
    chain = torch.zeros((len(path) - 1, 2, x_out.shape[1]), device=x_out.device)
    for idx, _ in enumerate(path):
        if idx != len(path) -1:
            chain[idx, 0] = x_out[path[idx]]
            chain[idx, 1] = x_out[path[idx+1]]
    return chain

flow_over_trees = 0
for prob, tree in trees:
    flow_over_paths = 0
    for target, cluster_prob in enumerate(P[target_cell, :])
        root = [n for n in tree.nodes if tree.in_degree(n) == 0]
        path = get_path_from_root(tree, root, target) 
        chain = get_chain_from_path(path, x_out)
        X = generate_integration_matrix(vf, chain)
        flow = torch.sum(X, dim=0)
        flow_over_paths += flow * cluster_prob
    flow_over_trees += flow_over_paths * prob




In [ ]:
def get_trees(A, P, root_cell):
    G = nx.from_numpy_array(A, create_using=nx.DiGraph)
    trees = []
    for root_cluster in range(P.shape(1)):
        prob = P[root_cell][root_cluster]
        for u in list(G.predecessors(root_cluster)):
            G.remove_edge(u, root_cluster)
        msa = nx.maximum_spanning_arborescence(G)
        trees.append((prob, msa))
    
    return trees
def get_path_from_root(tree, root, target):
    msa = tree[1]
    path = [target]
    current = target
    
    while current != root:
        preds = list(msa.predecessors(current))
        
        if not preds:
            raise ValueError("Target not reachable from root")
        
        current = preds[0]
        path.append(current)
    
    path.reverse()
    return path
def get_chain_from_path(path, x_out):
    chain = torch.zeros((len(path) - 1, 2, x_out.shape[1]), device=x_out.device)
    for idx, _ in enumerate(path):
        if idx != len(path) -1:
            chain[idx, 0] = x_out[path[idx]]
            chain[idx, 1] = x_out[path[idx+1]]
    return chain
def get_flow(trees, target_cell, P, x_out, vf):
    flow_over_trees = 0.0
    for prob, tree in trees:
        flow_over_paths = 0.0
        for target, cluster_prob in enumerate(P[target_cell, :]):
            root = [n for n in tree.nodes if tree.in_degree(n) == 0]
            root = root[0]
            path = get_path_from_root(tree, root, target) 
            chain = get_chain_from_path(path, x_out)
            X = generate_integration_matrix(vf, chain)
            flow = torch.sum(X, dim=0)
            flow_over_paths += flow * cluster_prob
    flow_over_trees += flow_over_paths * prob
    return flow_over_trees


trees = get_trees(A, P, root_cell)
flows = torch.zeros(A.shape(0))
for i in range(A.shape(0)):
    flow = get_flow(trees, i, P, x_out, vf)
    flows[i] = flow

distances = (node_embeddings - node_embeddings[root_cell]) ** 2
pseudotimes = distances * torch.exp(-flows)